In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("..")

In [3]:
from IPython.display import clear_output
from src.dataset_loaders import load_vectors, get_samplers
from src.utils import get_pca_models
from src import utils
from src.train import train_discrete
import wandb
from torch.utils.data import TensorDataset, DataLoader
import yaml
import numpy as np
import random
import pickle
import gc

## 1. Parameters.

Possible ```DATASET_NAME``` values are: ```twitter```, ```wiki-gigaword```, ```bone_marrow``` and ```muse```

In [4]:
DATASET_NAME = 'muse_multi'

SOURCE_DIM   = 100  
TARGET_DIM   = 100

EMB_TYPE_SOURCE = 'BP'
EMB_TYPE_TARGET = 'BP'

SOURCE_LANG     = 'en'
TARGET_LANG     = 'es'

MAX_ITERS    = 100
VS           = 200000

In [5]:

METHOD_NAME  = 'AlignGW'
DEVICE       = 'cpu'

ALPHA_INIT    = 1.0
SEED_INIT     = 43
COST_DISCRETE = 'cosine'

config = {'dataset':dict(DATASET_NAME     = DATASET_NAME,
                         DEVICE           = DEVICE,
                         SOURCE_DIM       = SOURCE_DIM,
                         TARGET_DIM       = TARGET_DIM,
                         EMB_TYPE_SOURCE  = EMB_TYPE_SOURCE,
                         EMB_TYPE_TARGET  = EMB_TYPE_TARGET,
                         VS               = VS,
                         SOURCE_LANG      = SOURCE_LANG,
                         TARGET_LANG      = TARGET_LANG,
                         
                         N_MAX_SAMPLES    = 60000, #set to 6667 to get N_train=3K
                         N_TRAIN_SAMPLES  = 6000, #We used 6000 for the others
                         N_TEST_SAMPLES   = 512,
                         N_EVAL           = 4,
                         ALPHA            = ALPHA_INIT, 
                         SEED             = SEED_INIT,
                         NORMALIZE_VECS   = False
                          ),
          
          'training':dict(TRAIN_TYPE           = 'discrete',
                          METHOD_NAME          = METHOD_NAME,
                          MAX_ITERS            = MAX_ITERS,
                          COST_DISCRETE        = COST_DISCRETE,
                          ),

          #===============================RegGW===============================
          #'model_specific':dict(HIDDEN_SIZES_MLP = [512, 256, 256],
          #                      EPS_FIT          = 0.01,
          #                      EPS_REG          = 0.001,
          #                      LAMBDA           = 1,
          #                      MOVER_LR         = 1e-4
          #                     ),

          #===============================FlowGW===============================
          #'model_specific':dict(HIDDEN_SIZES_MLP = [1024, 1024, 1024, 1024],
          #                      EPS              = 1e-4,
          #                      N_FREQ           = 128,
          #                      MOVER_LR         = 1e-4,
          #                      )
          
          #===============================AlignGW===============================
          #===============================StructuredGW===============================
          
          'model_specific':dict(EPS = 1e-4,
                               ),
         }


## 2. Loading dataset.

In [6]:
dataset_path = '../datasets'
sys.path.append(dataset_path)

source_vectors, target_vectors = load_vectors(dataset_path, config)

print(source_vectors.shape)
print(target_vectors.shape)


torch.Size([60000, 100])
torch.Size([60000, 100])


## 3. Training.

In [ ]:
import os
#os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'

n_repeats = 10
wandb_report = True

N            = config['dataset']['N_MAX_SAMPLES']//1000
TRAIN_TYPE   = config['training']['TRAIN_TYPE']
#project_name = f'{METHOD_NAME}_{DATASET_NAME}_{SOURCE_DIM}({EMB_TYPE_SOURCE})->{TARGET_DIM}({EMB_TYPE_TARGET})_{N}K_{n_repeats}reps_vs({VS//1000}K)'
project_name = f'{METHOD_NAME}_{DATASET_NAME}_{SOURCE_DIM}({SOURCE_LANG}_{EMB_TYPE_SOURCE})->{TARGET_DIM}({TARGET_LANG}_{EMB_TYPE_TARGET})_{N}K_{n_repeats}reps_vs({VS//1000}K)_final'

metrics_names = ['Top@1', 'Top@5', 'Top@10', 'cossim_gt', 'inner_gw', 'foscttm']

_, _, _, _, test_sampler = get_samplers(config, source_vectors, target_vectors)      

alpha_values = [0.0][::-1]

metrics_out = {str(np.round(alpha, 1)):[] for alpha in alpha_values}

for ALPHA in alpha_values:
    
    config['dataset']['ALPHA'] = ALPHA 
    
        
    print('================================')
    print(f'Experiment for ALPHA={ALPHA}')
    print('================================')
    
    for ix in range(n_repeats):
        
        if wandb_report:
            exp_name = f'ALPHA_{np.round(ALPHA, 1)}_repeat_{ix}'
            wandb.init(name=exp_name, config=config, project=project_name)
            
        SEED = random.randint(0, 10000)
        config['dataset']['SEED'] = SEED
        print('Seed: ', SEED)
        
        source_vectors, target_vectors, train_source_sampler, train_target_sampler, _ = get_samplers(config, source_vectors, target_vectors) 
        
        trained_class, metrics_dict = train_discrete(train_source_sampler, train_target_sampler,
                                                     test_sampler, 
                                                     metrics_names, target_vectors,
                                                     config,
                                                     wandb_report=wandb_report,
                                                     axis_lims=None, report_every=20)
        
        metrics_out[str(np.round(ALPHA, 1))].append(metrics_dict)

        with open(f'results_{TRAIN_TYPE}_final/{project_name}_00.pkl', 'wb') as f:
            pickle.dump(metrics_out, f)

        gc.collect()

Source pairs...
3000
tensor([ 2526, 18748, 45627,  ..., 56540,  6747, 34050], dtype=torch.int32)
Target pairs...
3000
tensor([ 2526, 18748, 45627,  ..., 56540,  6747, 34050], dtype=torch.int32)
Experiment for ALPHA=0.0


wandb: Currently logged in as: xavier13091994 (entropic_gw). Use `wandb login --relogin` to force relogin


Seed:  5553
Source pairs...
0
tensor([], dtype=torch.int32)
Target pairs...
0
tensor([], dtype=torch.int32)


  0%|          | 0/100 [00:00<?, ?it/s]